## **Random Forest Hyperparameters**

### **Topic Roadmap**

**1. Prepare a classification dataset**

**2. Understand core hyperparameters**

**3. Tune a compact search space**

**4. Evaluate the selected model**

**5. Key revision notes**

## **1. Dataset**

Hyperparameter experiments should use a fixed split and a fixed random state so changes can be attributed to the model settings.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=RANDOM_STATE
)

## **2. Core Hyperparameters**

`n_estimators` controls the number of trees. `max_depth`, `min_samples_leaf`, and `max_features` control tree complexity and diversity. `class_weight` can address class imbalance.

In [2]:
baseline = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
baseline.fit(X_train, y_train)
print(f"Baseline accuracy: {baseline.score(X_test, y_test):.3f}")

Baseline accuracy: 0.956


## **3. Randomized Hyperparameter Search**

Randomized search explores a bounded set of settings without evaluating every possible combination.

In [3]:
param_distributions = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"],
}
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_distributions, n_iter=20, cv=5,
    scoring="accuracy", random_state=RANDOM_STATE, n_jobs=-1
)
search.fit(X_train, y_train)
print(search.best_params_)
print(f"Best CV accuracy: {search.best_score_:.3f}")

{'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 5, 'class_weight': 'balanced'}
Best CV accuracy: 0.963


In [4]:
predictions = search.best_estimator_.predict(X_test)
print(classification_report(y_test, predictions, target_names=data.target_names))

              precision    recall  f1-score   support

   malignant       0.91      0.93      0.92        42
      benign       0.96      0.94      0.95        72

    accuracy                           0.94       114
   macro avg       0.93      0.94      0.93       114
weighted avg       0.94      0.94      0.94       114



### **Key Revision Notes**

- `n_estimators` mainly affects stability and computation.
- Depth and leaf-size parameters control individual-tree variance.
- `max_features` creates diversity among trees.
- Tune with cross-validation and report performance on a held-out test set.